## Name: Reem Alyahya

# M4.Ex2: Penguins Classification (PyCaret)

- Run: [**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/B5/blob/main/content/W3/M6/exercises/ex2_pycaret_classification.ipynb)

## Exercise

Your task is to follow the steps outlined here, and apply them on the **Palmer Penguins Dataset** below:

- [**🚀 Classification**](https://pycaret.gitbook.io/docs/get-started/quickstart#classification)
    - Setup
    - Compare Models
    - Analyze Model
    - Predictions
    - Save the model

## Palmer Penguins Dataset

The goal of palmer penguins is to provide a great dataset for data exploration & visualization, as an alternative to iris.

The data contains 344 penguins. There are 3 different species of penguins in this dataset, collected from 3 islands in the Palmer Archipelago, Antarctica.

- Features: `4` numerical, `2` categorical
- Target: `species` (Categorical / 3 classes)
- Size: `344` samples
- Source: [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/)

### Load the data

In [1]:
import seaborn as sns
import MultilabelPredictor as mlp
from autogluon.tabular import TabularDataset, TabularPredictor

penguins = sns.load_dataset('penguins')
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## Experiments

1. **First experiment**:
    - X = Flipper Length (numerical) & Bill Length (numerical)
    - y = Species (categorical)
2. **Second experiment**:
    - X = Weights (numerical) & Species (categorical)
    - y = Sex (categroical)
3. **Third experiment**:
    - X = `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`
    - y = Sex and Species (multi-label classification)

In [35]:
penguins = penguins.dropna(subset=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex'])

In [38]:
#Data and Target for experiment 1:

data_ex1 = penguins[['flipper_length_mm', 'bill_length_mm', 'species']]
#target_1 = penguins['species']

In [39]:
#Data and Target for experiment 2:
data_ex2 = penguins[['body_mass_g', 'species', 'sex']]
#target_2 = penguins['sex']

In [40]:
#Data and Target for experiment 3:
data_ex3 = penguins[['island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
target_3 = penguins[['sex', 'species']]

### Experiment #1:

In [5]:
from sklearn.model_selection import train_test_split

train_data1, test_data1 = train_test_split(
    data_ex1,
    test_size=0.2,
    random_state=42
    )


In [6]:
custom_hyperparameters = {
    'RF': {},   # Random Forest
    'CAT': {},  # CatBoost
    #'XGB': {},  # XGBoost
    'FASTAI':{}, #NNFastAiTabularModel
    'NN_TORCH':{},
    # 'GBM': {}, # <-- Omit this to exclude LightGBM
}

predictor = TabularPredictor( label= 'species' ).fit(
    train_data1,
    presets='medium_quality_faster_train',
    time_limit= 60,
    hyperparameters=custom_hyperparameters,
    included_model_types = ['FASTAI','NN_TORCH','RF','CAT'],
    num_cpus= 2,
    ag_args_fit ={'num_gpus' : 0}
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260423_041556"
Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   x86_64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Jul 14 11:28:17 PDT 2025; root:xnu-11417.140.69~1/RELEASE_X86_64
CPU Count:          16
Memory Avail:       5.36 GB / 16.00 GB (33.5%)
Disk Space Avail:   772.07 GB / 931.55 GB (82.9%)
Presets specified: ['medium_quality_faster_train']
Beginning AutoGluon training ... Time limit = 60s
AutoGluon will save models to "/Users/reemalyahya/Desktop/Bootcamp python/B5/student/C3/AutogluonModels/ag-20260423_041556"
Train Data Rows:    275
Train Data Columns: 2
Label Column:       species
AutoGluon infers your prediction problem is: 'multiclass' (because dtype of label-column == object).
	3 uniqu

In [ ]:
predictor.leaderboard() 

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost,0.963636,accuracy,0.000611,0.327443,0.000611,0.327443,1,True,2
1,WeightedEnsemble_L2,0.963636,accuracy,0.001263,0.365630,0.000652,0.038187,2,True,4
2,NeuralNetTorch,0.963636,accuracy,0.002985,2.408101,0.002985,2.408101,1,True,3
3,RandomForest,0.945455,accuracy,0.039261,0.581439,0.039261,0.581439,1,True,1


In [8]:
predictor.fit_summary(show_plot=True, verbosity=2) 

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0             CatBoost   0.963636    accuracy       0.000611  0.327443                0.000611           0.327443            1       True          2
1  WeightedEnsemble_L2   0.963636    accuracy       0.001263  0.365630                0.000652           0.038187            2       True          4
2       NeuralNetTorch   0.963636    accuracy       0.002985  2.408101                0.002985           2.408101            1       True          3
3         RandomForest   0.945455    accuracy       0.039261  0.581439                0.039261           0.581439            1       True          1
Number of models trained: 4
Types of models trained:
{'TabularNeuralNetTorchModel', 'WeightedEnsembleModel', 'RFModel', 'CatBoostModel'}
Bagging used: False 
Multi-layer stack-ensembling used: 

{'model_types': {'RandomForest': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'RandomForest': 0.9454545454545454,
  'CatBoost': 0.9636363636363636,
  'NeuralNetTorch': 0.9636363636363636,
  'WeightedEnsemble_L2': 0.9636363636363636},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'RandomForest': ['RandomForest'],
  'CatBoost': ['CatBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'WeightedEnsemble_L2': ['WeightedEnsemble_L2']},
 'model_fit_times': {'RandomForest': 0.5814387798309326,
  'CatBoost': 0.32744288444519043,
  'NeuralNetTorch': 2.4081010818481445,
  'WeightedEnsemble_L2': 0.03818678855895996},
 'model_pred_times': {'RandomForest': 0.0392608642578125,
  'CatBoost': 0.0006110668182373047,
  'NeuralNetTorch': 0.0029850006103515625,
  'WeightedEnsemble_L2': 0.0006520748138427734},
 'num_bag_folds': 0,
 'max_stack_level': 2,
 'num_classes': 3,
 'model_

### Predict and Evaliuat model1

In [9]:
predictor.predict(test_data1) 

194    Chinstrap
157    Chinstrap
225       Gentoo
208    Chinstrap
318    Chinstrap
         ...    
321       Gentoo
172       Adelie
73     Chinstrap
76        Adelie
16        Adelie
Name: species, Length: 69, dtype: object

In [10]:
import pandas as pd

In [11]:
predictor.evaluate(test_data1, detailed_report= True)

{'accuracy': 0.9565217391304348,
 'balanced_accuracy': 0.9528769841269842,
 'mcc': 0.9323266919409712,
 'confusion_matrix':            Adelie  Chinstrap  Gentoo
 Adelie         31          1       0
 Chinstrap       1         15       0
 Gentoo          0          1      20,
 'classification_report': {'Adelie': {'precision': 0.96875,
   'recall': 0.96875,
   'f1-score': 0.96875,
   'support': 32.0},
  'Chinstrap': {'precision': 0.8823529411764706,
   'recall': 0.9375,
   'f1-score': 0.9090909090909091,
   'support': 16.0},
  'Gentoo': {'precision': 1.0,
   'recall': 0.9523809523809523,
   'f1-score': 0.975609756097561,
   'support': 21.0},
  'accuracy': 0.9565217391304348,
  'macro avg': {'precision': 0.9503676470588235,
   'recall': 0.9528769841269842,
   'f1-score': 0.95115022172949,
   'support': 69.0},
  'weighted avg': {'precision': 0.958226768968457,
   'recall': 0.9565217391304348,
   'f1-score': 0.9570037597609178,
   'support': 69.0}}}

In [12]:
predictor.leaderboard(test_data1) 

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForest,0.971014,0.945455,accuracy,0.155200,0.039261,0.581439,0.155200,0.039261,0.581439,1,True,1
1,CatBoost,0.956522,0.963636,accuracy,0.002428,0.000611,0.327443,0.002428,0.000611,0.327443,1,True,2
2,WeightedEnsemble_L2,0.956522,0.963636,accuracy,0.003958,0.001263,0.365630,0.001530,0.000652,0.038187,2,True,4
3,NeuralNetTorch,0.942029,0.963636,accuracy,0.006986,0.002985,2.408101,0.006986,0.002985,2.408101,1,True,3


In [13]:
predictor.feature_importance(test_data1)

Computing feature importance via permutation shuffling for 2 features using 69 rows with 5 shuffle sets...
	0.08s	= Expected runtime (0.02s per shuffle set)
	0.04s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
flipper_length_mm,0.376812,0.042253,0.000019,5,0.463812,0.289812
bill_length_mm,0.342029,0.040476,0.000023,5,0.425370,0.258688


In [14]:
predictor.predict_proba(test_data1)

,Adelie,Chinstrap,Gentoo
194,0.254897,0.486929,0.258174
157,0.317202,0.423286,0.259512
225,0.249442,0.259743,0.490815
208,0.310266,0.438128,0.251606
318,0.254929,0.401647,0.343424
...,...,...,...
321,0.233773,0.247635,0.518593
172,0.477554,0.290361,0.232085
73,0.293239,0.443042,0.263719
76,0.564786,0.223587,0.211626


### model1 Deployment

In [ ]:
predictor.clone_for_deployment(path='ag-20260423_030505')

In [16]:
predictor_opt = TabularPredictor.load('ag-20260423_030505')

In [17]:
predictor_opt.predict(test_data1)

194    Chinstrap
157    Chinstrap
225       Gentoo
208    Chinstrap
318    Chinstrap
         ...    
321       Gentoo
172       Adelie
73     Chinstrap
76        Adelie
16        Adelie
Name: species, Length: 69, dtype: object

### Experiment #2:

In [52]:
from sklearn.model_selection import train_test_split

train_data2, test_data2 = train_test_split(
    data_ex2,
    test_size=0.2,
    random_state=42
    )
train_data2 = train_data2.dropna(subset='sex')


In [53]:
custom_hyperparameters = {
    'RF': {},   # Random Forest
    'CAT': {},  # CatBoost
    #'XGB': {},  # XGBoost
    'FASTAI':{}, #NNFastAiTabularModel
    'NN_TORCH':{},
    # 'GBM': {}, # <-- Omit this to exclude LightGBM
}

predictor2 = TabularPredictor( label= 'sex' ).fit(
    train_data2,
    presets='medium_quality_faster_train',
    time_limit= 60,
    hyperparameters=custom_hyperparameters,
    included_model_types = ['FASTAI','NN_TORCH','RF','CAT'],
    num_cpus= 2,
    ag_args_fit ={'num_gpus' : 0}
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260423_093357"
Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   x86_64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Jul 14 11:28:17 PDT 2025; root:xnu-11417.140.69~1/RELEASE_X86_64
CPU Count:          16
Memory Avail:       5.22 GB / 16.00 GB (32.6%)
Disk Space Avail:   771.02 GB / 931.55 GB (82.8%)
Presets specified: ['medium_quality_faster_train']
Beginning AutoGluon training ... Time limit = 60s
AutoGluon will save models to "/Users/reemalyahya/Desktop/Bootcamp python/B5/student/C3/AutogluonModels/ag-20260423_093357"
Train Data Rows:    266
Train Data Columns: 2
Label Column:       sex
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique 

In [54]:
predictor2.leaderboard() 

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,NeuralNetTorch,0.833333,accuracy,0.004435,0.457741,0.004435,0.457741,1,True,3
1,WeightedEnsemble_L2,0.833333,accuracy,0.005299,0.496664,0.000864,0.038923,2,True,4
2,CatBoost,0.814815,accuracy,0.000876,0.281709,0.000876,0.281709,1,True,2
3,RandomForest,0.759259,accuracy,0.040305,0.439141,0.040305,0.439141,1,True,1


In [55]:
predictor2.fit_summary(show_plot=True, verbosity=2) 

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0       NeuralNetTorch   0.833333    accuracy       0.004435  0.457741                0.004435           0.457741            1       True          3
1  WeightedEnsemble_L2   0.833333    accuracy       0.005299  0.496664                0.000864           0.038923            2       True          4
2             CatBoost   0.814815    accuracy       0.000876  0.281709                0.000876           0.281709            1       True          2
3         RandomForest   0.759259    accuracy       0.040305  0.439141                0.040305           0.439141            1       True          1
Number of models trained: 4
Types of models trained:
{'TabularNeuralNetTorchModel', 'WeightedEnsembleModel', 'RFModel', 'CatBoostModel'}
Bagging used: False 
Multi-layer stack-ensembling used: 

{'model_types': {'RandomForest': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'RandomForest': 0.7592592592592593,
  'CatBoost': 0.8148148148148148,
  'NeuralNetTorch': 0.8333333333333334,
  'WeightedEnsemble_L2': 0.8333333333333334},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'RandomForest': ['RandomForest'],
  'CatBoost': ['CatBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'WeightedEnsemble_L2': ['WeightedEnsemble_L2']},
 'model_fit_times': {'RandomForest': 0.43914127349853516,
  'CatBoost': 0.2817091941833496,
  'NeuralNetTorch': 0.45774102210998535,
  'WeightedEnsemble_L2': 0.038923025131225586},
 'model_pred_times': {'RandomForest': 0.04030489921569824,
  'CatBoost': 0.0008759498596191406,
  'NeuralNetTorch': 0.004434823989868164,
  'WeightedEnsemble_L2': 0.0008640289306640625},
 'num_bag_folds': 0,
 'max_stack_level': 2,
 'num_classes': 2,
 'mode

In [56]:
predictor2.predict(test_data2) 

30     Female
317    Female
79       Male
201    Female
63       Male
        ...  
288    Female
4      Female
83       Male
319      Male
66     Female
Name: sex, Length: 67, dtype: object

In [57]:
predictor2.evaluate(test_data2, detailed_report= True)

{'accuracy': 0.8507462686567164,
 'balanced_accuracy': 0.8522522522522522,
 'mcc': 0.7013523432591564,
 'roc_auc': 0.9382882882882881,
 'f1': 0.8387096774193549,
 'precision': 0.8125,
 'recall': 0.8666666666666667,
 'confusion_matrix':         Female  Male
 Female      31     6
 Male         4    26,
 'classification_report': {'Female': {'precision': 0.8857142857142857,
   'recall': 0.8378378378378378,
   'f1-score': 0.8611111111111112,
   'support': 37.0},
  'Male': {'precision': 0.8125,
   'recall': 0.8666666666666667,
   'f1-score': 0.8387096774193549,
   'support': 30.0},
  'accuracy': 0.8507462686567164,
  'macro avg': {'precision': 0.8491071428571428,
   'recall': 0.8522522522522522,
   'f1-score': 0.8499103942652331,
   'support': 67.0},
  'weighted avg': {'precision': 0.8529317697228145,
   'recall': 0.8507462686567164,
   'f1-score': 0.8510806184133098,
   'support': 67.0}}}

In [58]:
predictor2.leaderboard(test_data2) 

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost,0.865672,0.814815,accuracy,0.003556,0.000876,0.281709,0.003556,0.000876,0.281709,1,True,2
1,RandomForest,0.865672,0.759259,accuracy,0.059312,0.040305,0.439141,0.059312,0.040305,0.439141,1,True,1
2,NeuralNetTorch,0.850746,0.833333,accuracy,0.006538,0.004435,0.457741,0.006538,0.004435,0.457741,1,True,3
3,WeightedEnsemble_L2,0.850746,0.833333,accuracy,0.007781,0.005299,0.496664,0.001243,0.000864,0.038923,2,True,4


In [59]:
predictor2.feature_importance(test_data2)

Computing feature importance via permutation shuffling for 2 features using 67 rows with 5 shuffle sets...
	0.2s	= Expected runtime (0.04s per shuffle set)
	0.17s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
body_mass_g,0.355224,0.024525,0.000003,5,0.405721,0.304727
species,0.134328,0.035003,0.000507,5,0.206400,0.062256


In [60]:
predictor2.predict_proba(test_data2)

,Female,Male
30,0.914912,0.085088
317,0.606539,0.393461
79,0.195343,0.804657
201,0.613829,0.386171
63,0.169950,0.830050
...,...,...
288,0.744712,0.255288
4,0.816416,0.183584
83,0.121321,0.878679
319,0.246708,0.753292


### Deployment model2:

In [62]:
predictor2.clone_for_deployment(path='ag-20260423_093357')

Cloned TabularPredictor located in '/Users/reemalyahya/Desktop/Bootcamp python/B5/student/C3/AutogluonModels/ag-20260423_093357' to 'ag-20260423_093357'.
	To load the cloned predictor: predictor_clone = TabularPredictor.load(path="ag-20260423_093357")


Clone: Keeping minimum set of models required to predict with best model 'WeightedEnsemble_L2'...
Deleting model RandomForest. All files under /Users/reemalyahya/Desktop/Bootcamp python/B5/student/C3/ag-20260423_093357/models/RandomForest will be removed.
Deleting model CatBoost. All files under /Users/reemalyahya/Desktop/Bootcamp python/B5/student/C3/ag-20260423_093357/models/CatBoost will be removed.
Clone: Removing artifacts unnecessary for prediction. NOTE: Clone can no longer fit new models, and most functionality except for predict and predict_proba will no longer work


'/Users/reemalyahya/Desktop/Bootcamp python/B5/student/C3/ag-20260423_093357'

In [63]:
predictor2_opt = TabularPredictor.load('ag-20260423_093357')

In [64]:
predictor2_opt.predict(test_data2)

30     Female
317    Female
79       Male
201    Female
63       Male
        ...  
288    Female
4      Female
83       Male
319      Male
66     Female
Name: sex, Length: 67, dtype: object